In [2]:

# ============================================================================
# OPTIMIZED PRICE PREDICTION MODEL - TRAINING PIPELINE (FIXED VERSION)
# ============================================================================

# STEP 1: Install Required Libraries
!pip install pandas numpy matplotlib seaborn scikit-learn joblib
!pip install xgboost lightgbm catboost

# ============================================================================
# STEP 2: Import Libraries
# ============================================================================

import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

# Check GPU availability
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], encoding='utf-8')
    print("\n✓ GPU DETECTED")
    GPU_AVAILABLE = True
except:
    print("\n⚠ No GPU detected. Using CPU mode.")
    GPU_AVAILABLE = False

# ============================================================================
# STEP 3: ENHANCED Feature Extraction Function (150+ Features)
# ============================================================================

def extract_enhanced_features(catalog_content):
    """
    Extract 150+ comprehensive features from catalog_content
    MUST BE IDENTICAL BETWEEN TRAINING AND PREDICTION
    """
    features = {}
    text_lower = catalog_content.lower()

    # ============ BASIC PRODUCT INFO ============
    item_name_match = re.search(r'Item Name:\s*(.+?)(?:\n|$)', catalog_content)
    features['item_name'] = item_name_match.group(1).strip() if item_name_match else ""

    # Extract brand with better parsing
    if features['item_name']:
        brand_patterns = [
            r'^([A-Z][A-Za-z0-9\'&\.\-\s]{2,30})(?:[,\(]|\s{2,})',
            r'^([A-Za-z0-9\'&\.\-\s]+?)(?:[,]|\s+-\s+)',
            r'^([A-Za-z0-9\'&\.\-\s]+?)\s+\d',
        ]
        features['brand'] = ""
        for pattern in brand_patterns:
            brand_match = re.match(pattern, features['item_name'])
            if brand_match:
                features['brand'] = brand_match.group(1).strip()
                break
    else:
        features['brand'] = ""

    features['brand_name_length'] = len(features['brand']) if features['brand'] else 0
    features['has_brand_name'] = 1 if features['brand'] else 0
    features['brand_word_count'] = len(features['brand'].split()) if features['brand'] else 0

    # ============ ADVANCED QUANTITY & SIZE FEATURES ============
    value_match = re.search(r'Value:\s*([\d\.]+)', catalog_content)
    features['value'] = float(value_match.group(1)) if value_match else 1.0

    unit_match = re.search(r'Unit:\s*(.+?)(?:\n|$)', catalog_content)
    features['unit'] = unit_match.group(1).strip() if unit_match else "Count"

    # Comprehensive unit type encoding
    unit_lower = features['unit'].lower()
    features['unit_is_oz'] = 1 if 'oz' in unit_lower or 'ounce' in unit_lower else 0
    features['unit_is_lb'] = 1 if 'lb' in unit_lower or 'pound' in unit_lower else 0
    features['unit_is_fl_oz'] = 1 if 'fl oz' in unit_lower or 'fluid ounce' in unit_lower else 0
    features['unit_is_gram'] = 1 if 'gram' in unit_lower or unit_lower == 'g' else 0
    features['unit_is_kg'] = 1 if 'kg' in unit_lower or 'kilogram' in unit_lower else 0
    features['unit_is_ml'] = 1 if 'ml' in unit_lower or 'milliliter' in unit_lower else 0
    features['unit_is_liter'] = 1 if 'liter' in unit_lower or unit_lower == 'l' else 0
    features['unit_is_count'] = 1 if 'count' in unit_lower else 0
    features['unit_is_gallon'] = 1 if 'gallon' in unit_lower or 'gal' in unit_lower else 0
    features['unit_is_quart'] = 1 if 'quart' in unit_lower or 'qt' in unit_lower else 0
    features['unit_is_pint'] = 1 if 'pint' in unit_lower or 'pt' in unit_lower else 0
    features['unit_is_cup'] = 1 if 'cup' in unit_lower else 0
    features['unit_is_tablespoon'] = 1 if 'tablespoon' in unit_lower or 'tbsp' in unit_lower else 0
    features['unit_is_teaspoon'] = 1 if 'teaspoon' in unit_lower or 'tsp' in unit_lower else 0

    # Pack size extraction
    pack_patterns = [
        r'\(Pack of (\d+)\)',
        r'Pack of (\d+)',
        r'(\d+)\s*Pack',
        r'(\d+)[\-\s]*pack',
        r'(\d+)[- ]ct',
        r'(\d+)\s*count'
    ]
    pack_size = 1
    for pattern in pack_patterns:
        pack_match = re.search(pattern, features['item_name'], re.IGNORECASE)
        if pack_match:
            pack_size = int(pack_match.group(1))
            break
    features['pack_size'] = pack_size
    features['is_multipack'] = 1 if pack_size > 1 else 0
    features['is_bulk_pack'] = 1 if pack_size >= 12 else 0

    case_match = re.search(r'(\d+)\s*per\s*case', text_lower)
    features['case_count'] = int(case_match.group(1)) if case_match else 1

    # Item size extraction
    size_patterns = [
        (r'(\d+\.?\d*)\s*(oz|ounce)s?(?!\s*\(pack)', 'oz'),
        (r'(\d+\.?\d*)\s*(lb|pound)s?', 'lb'),
        (r'(\d+\.?\d*)\s*g(?!\s*\w)', 'g'),
        (r'(\d+\.?\d*)\s*(kg|kilogram)s?', 'kg'),
        (r'(\d+\.?\d*)\s*(ml|milliliter)s?', 'ml'),
        (r'(\d+\.?\d*)\s*(l|liter)s?', 'l'),
        (r'(\d+\.?\d*)\s*fl\s*oz', 'fl_oz'),
        (r'(\d+\.?\d*)\s*(gal|gallon)s?', 'gal'),
        (r'(\d+\.?\d*)\s*(qt|quart)s?', 'qt')
    ]

    item_size = 0
    item_size_unit = ""
    for pattern, unit_name in size_patterns:
        size_match = re.search(pattern, features['item_name'], re.IGNORECASE)
        if size_match:
            item_size = float(size_match.group(1))
            try:
                item_size_unit = size_match.group(2).lower()
            except IndexError:
                item_size_unit = unit_name
            break

    features['item_size'] = item_size
    features['item_size_unit'] = item_size_unit
    features['has_size_info'] = 1 if item_size > 0 else 0

    # Total quantity calculations
    features['total_quantity'] = features['value'] * features['pack_size']
    features['total_weight_oz'] = features['value'] * features['pack_size'] if features['unit_is_oz'] else 0

    # Log and power transforms
    features['log_value'] = np.log1p(features['value'])
    features['log_pack_size'] = np.log1p(features['pack_size'])
    features['log_total_quantity'] = np.log1p(features['total_quantity'])
    features['sqrt_value'] = np.sqrt(features['value'])
    features['value_squared'] = features['value'] ** 2
    features['value_per_pack'] = features['value'] / features['pack_size'] if features['pack_size'] > 0 else features['value']

    # ============ DESCRIPTION QUALITY FEATURES ============
    bullet_points = re.findall(r'Bullet Point \d+:\s*(.+?)(?:\n|$)', catalog_content)
    features['num_bullet_points'] = len(bullet_points)
    features['bullet_points_text'] = ' '.join(bullet_points) if bullet_points else ""

    prod_desc_match = re.search(r'Product Description:\s*(.+?)(?:\nValue:|$)', catalog_content, re.DOTALL)
    features['product_description'] = prod_desc_match.group(1).strip() if prod_desc_match else ""
    features['has_product_description'] = 1 if prod_desc_match else 0

    # Text lengths
    features['description_length'] = len(catalog_content)
    features['bullet_points_length'] = len(features['bullet_points_text'])
    features['product_desc_length'] = len(features['product_description'])
    features['item_name_length'] = len(features['item_name'])
    features['item_name_word_count'] = len(features['item_name'].split())

    features['avg_bullet_length'] = (features['bullet_points_length'] / features['num_bullet_points']
                                     if features['num_bullet_points'] > 0 else 0)
    features['min_bullet_length'] = min([len(bp) for bp in bullet_points]) if bullet_points else 0
    features['max_bullet_length'] = max([len(bp) for bp in bullet_points]) if bullet_points else 0

    features['total_word_count'] = len(catalog_content.split())
    features['unique_word_ratio'] = len(set(text_lower.split())) / len(text_lower.split()) if text_lower.split() else 0
    features['description_to_name_ratio'] = features['description_length'] / features['item_name_length'] if features['item_name_length'] > 0 else 0

    features['log_description_length'] = np.log1p(features['description_length'])
    features['log_item_name_length'] = np.log1p(features['item_name_length'])
    features['log_total_word_count'] = np.log1p(features['total_word_count'])

    # ============ DIETARY ATTRIBUTES ============
    features['is_organic'] = 1 if re.search(r'\b(organic|usda organic|certified organic)\b', text_lower) else 0
    features['is_gluten_free'] = 1 if re.search(r'\bgluten[- ]free\b', text_lower) else 0
    features['is_vegan'] = 1 if re.search(r'\bvegan\b', text_lower) else 0
    features['is_vegetarian'] = 1 if re.search(r'\bvegetarian\b', text_lower) else 0
    features['is_kosher'] = 1 if re.search(r'\bkosher\b', text_lower) else 0
    features['is_halal'] = 1 if re.search(r'\bhalal\b', text_lower) else 0
    features['is_non_gmo'] = 1 if re.search(r'\bnon[- ]gmo\b', text_lower) else 0
    features['is_sugar_free'] = 1 if re.search(r'\b(sugar[- ]free|no sugar|zero sugar|unsweetened)\b', text_lower) else 0
    features['is_dairy_free'] = 1 if re.search(r'\b(dairy[- ]free|lactose free)\b', text_lower) else 0
    features['is_natural'] = 1 if re.search(r'\b(natural|all natural|100% natural)\b', text_lower) else 0
    features['is_keto'] = 1 if re.search(r'\b(keto|ketogenic|keto[- ]friendly)\b', text_lower) else 0
    features['is_paleo'] = 1 if re.search(r'\b(paleo|paleo[- ]friendly)\b', text_lower) else 0
    features['is_low_carb'] = 1 if re.search(r'\b(low carb|low[- ]carbohydrate)\b', text_lower) else 0
    features['is_high_protein'] = 1 if re.search(r'\b(high protein|protein packed|protein rich)\b', text_lower) else 0
    features['is_high_fiber'] = 1 if re.search(r'\b(high fiber|fiber rich)\b', text_lower) else 0
    features['is_low_fat'] = 1 if re.search(r'\b(low fat|reduced fat)\b', text_lower) else 0
    features['is_fat_free'] = 1 if re.search(r'\b(fat[- ]free|no fat|0g fat|zero fat)\b', text_lower) else 0
    features['is_low_sodium'] = 1 if re.search(r'\b(low sodium|reduced sodium|no salt added)\b', text_lower) else 0
    features['is_low_calorie'] = 1 if re.search(r'\b(low calorie|low cal|diet)\b', text_lower) else 0
    features['is_whole30'] = 1 if re.search(r'\bwhole30\b', text_lower) else 0
    features['is_nut_free'] = 1 if re.search(r'\b(nut[- ]free|no nuts)\b', text_lower) else 0
    features['is_soy_free'] = 1 if re.search(r'\b(soy[- ]free|no soy)\b', text_lower) else 0
    features['is_wheat_free'] = 1 if re.search(r'\b(wheat[- ]free|no wheat)\b', text_lower) else 0
    features['is_egg_free'] = 1 if re.search(r'\b(egg[- ]free|no eggs)\b', text_lower) else 0

    dietary_attrs = [
        features['is_organic'], features['is_gluten_free'], features['is_vegan'],
        features['is_kosher'], features['is_halal'], features['is_non_gmo'],
        features['is_sugar_free'], features['is_dairy_free'], features['is_natural'],
        features['is_keto'], features['is_paleo'], features['is_low_carb'],
        features['is_high_protein'], features['is_high_fiber'], features['is_whole30']
    ]
    features['total_dietary_claims'] = sum(dietary_attrs)
    features['has_any_dietary_claim'] = 1 if features['total_dietary_claims'] > 0 else 0
    features['has_multiple_dietary_claims'] = 1 if features['total_dietary_claims'] >= 3 else 0

    # ============ PRODUCT CATEGORIES ============
    features['is_food'] = 1 if re.search(r'\b(food|edible|eat)\b', text_lower) else 0
    features['is_beverage'] = 1 if re.search(r'\b(drink|beverage|juice|water|tea|coffee|soda|wine|beer|liquor)\b', text_lower) else 0
    features['is_snack'] = 1 if re.search(r'\b(snack|chip|cookie|cracker|candy|gum|popcorn|pretzel)\b', text_lower) else 0
    features['is_spice_seasoning'] = 1 if re.search(r'\b(spice|seasoning|salt|pepper|herb|blend)\b', text_lower) else 0
    features['is_sauce_condiment'] = 1 if re.search(r'\b(sauce|marinade|dressing|ketchup|mustard|mayo|condiment|salsa)\b', text_lower) else 0
    features['is_baking'] = 1 if re.search(r'\b(flour|sugar|baking|mix|dough|yeast)\b', text_lower) else 0
    features['is_cereal'] = 1 if re.search(r'\b(cereal|granola|oats|oatmeal|muesli)\b', text_lower) else 0
    features['is_pasta_grain'] = 1 if re.search(r'\b(pasta|rice|grain|noodle|couscous|quinoa)\b', text_lower) else 0
    features['is_nuts_seeds'] = 1 if re.search(r'\b(nut|almond|cashew|peanut|seed|sunflower|walnut|pecan|pistachio)\b', text_lower) else 0
    features['is_protein_bar'] = 1 if re.search(r'\b(protein bar|energy bar|snack bar|nutrition bar|meal bar)\b', text_lower) else 0
    features['is_chocolate'] = 1 if re.search(r'\b(chocolate|cocoa|cacao)\b', text_lower) else 0
    features['is_candy'] = 1 if re.search(r'\b(candy|gummy|lollipop|taffy|mint|caramel)\b', text_lower) else 0
    features['is_coffee_tea'] = 1 if re.search(r'\b(coffee|tea|espresso|latte|cappuccino|matcha)\b', text_lower) else 0
    features['is_oil_vinegar'] = 1 if re.search(r'\b(oil|olive oil|vinegar|coconut oil|vegetable oil|avocado oil)\b', text_lower) else 0
    features['is_beans_legumes'] = 1 if re.search(r'\b(bean|beans|lentil|chickpea|pea|legume)\b', text_lower) else 0
    features['is_jam_spread'] = 1 if re.search(r'\b(jam|jelly|spread|marmalade|preserve|butter|nutella)\b', text_lower) else 0
    features['is_meat_seafood'] = 1 if re.search(r'\b(meat|beef|chicken|pork|fish|seafood|salmon|tuna|jerky)\b', text_lower) else 0
    features['is_soup'] = 1 if re.search(r'\b(soup|broth|stock|chowder|stew)\b', text_lower) else 0
    features['is_dairy'] = 1 if re.search(r'\b(milk|cheese|yogurt|butter|cream|dairy)\b', text_lower) else 0
    features['is_fruit'] = 1 if re.search(r'\b(fruit|apple|banana|berry|orange|grape)\b', text_lower) else 0
    features['is_vegetable'] = 1 if re.search(r'\b(vegetable|veggie|carrot|broccoli|tomato)\b', text_lower) else 0
    features['is_bread_bakery'] = 1 if re.search(r'\b(bread|bagel|muffin|roll|croissant|biscuit)\b', text_lower) else 0
    features['is_supplement'] = 1 if re.search(r'\b(supplement|vitamin|mineral|capsule|tablet)\b', text_lower) else 0
    features['is_baby_food'] = 1 if re.search(r'\b(baby food|infant|toddler)\b', text_lower) else 0
    features['is_pet_food'] = 1 if re.search(r'\b(pet food|dog food|cat food|pet treat)\b', text_lower) else 0
    features['is_alcohol'] = 1 if re.search(r'\b(wine|beer|whiskey|vodka|rum|alcohol|liquor)\b', text_lower) else 0
    features['is_water'] = 1 if re.search(r'\b(water|sparkling water|mineral water)\b', text_lower) else 0
    features['is_soda_soft_drink'] = 1 if re.search(r'\b(soda|cola|pop|soft drink|carbonated)\b', text_lower) else 0
    features['is_juice'] = 1 if re.search(r'\b(juice|fruit juice|orange juice)\b', text_lower) else 0
    features['is_energy_drink'] = 1 if re.search(r'\b(energy drink|sports drink)\b', text_lower) else 0

    category_count = sum([
        features['is_beverage'], features['is_snack'], features['is_spice_seasoning'],
        features['is_sauce_condiment'], features['is_baking'], features['is_cereal'],
        features['is_pasta_grain'], features['is_nuts_seeds'], features['is_chocolate']
    ])
    features['category_count'] = category_count

    # ============ QUALITY INDICATORS ============
    features['has_premium_keywords'] = 1 if re.search(r'\b(premium|gourmet|quality|finest|artisan|luxury|deluxe|elite)\b', text_lower) else 0
    features['has_bulk_keywords'] = 1 if re.search(r'\b(bulk|pack|case|count|variety|assorted)\b', text_lower) else 0
    features['mentions_value'] = 1 if re.search(r'\b(value|save|deal|affordable|budget|economy|discount)\b', text_lower) else 0
    features['has_award'] = 1 if re.search(r'\b(award|winner|certified|rated|best|top)\b', text_lower) else 0
    features['is_imported'] = 1 if re.search(r'\b(import|imported|product of)\b', text_lower) else 0
    features['made_in_usa'] = 1 if re.search(r'\b(made in usa|made in america|usa made|american made)\b', text_lower) else 0
    features['is_handmade'] = 1 if re.search(r'\b(handmade|hand[- ]crafted|artisan|craft)\b', text_lower) else 0
    features['is_family_owned'] = 1 if re.search(r'\b(family[- ]owned|family business)\b', text_lower) else 0
    features['has_tradition'] = 1 if re.search(r'\b(traditional|since \d{4}|heritage|original|classic|authentic)\b', text_lower) else 0
    features['is_sustainable'] = 1 if re.search(r'\b(sustainable|eco[- ]friendly|green|environmentally)\b', text_lower) else 0
    features['is_fresh'] = 1 if re.search(r'\b(fresh|freshly|farm fresh)\b', text_lower) else 0
    features['is_local'] = 1 if re.search(r'\b(local|locally sourced|regional)\b', text_lower) else 0
    features['is_small_batch'] = 1 if re.search(r'\b(small batch|craft|limited)\b', text_lower) else 0
    features['has_certification'] = 1 if re.search(r'\b(certified|certification|usda|fda)\b', text_lower) else 0

    quality_indicators = [
        features['has_premium_keywords'], features['has_award'], features['made_in_usa'],
        features['is_handmade'], features['is_sustainable'], features['has_tradition'],
        features['is_family_owned'], features['has_certification']
    ]
    features['quality_score'] = sum(quality_indicators)
    features['is_premium_product'] = 1 if features['quality_score'] >= 2 else 0

    # ============ CONVENIENCE ============
    features['is_ready_to_eat'] = 1 if re.search(r'\b(ready to eat|no prep|instant|ready)\b', text_lower) else 0
    features['is_frozen'] = 1 if re.search(r'\b(frozen|freeze[- ]dried)\b', text_lower) else 0
    features['is_dried'] = 1 if re.search(r'\b(dried|dehydrated)\b', text_lower) else 0
    features['is_canned'] = 1 if re.search(r'\b(canned|can)\b', text_lower) else 0
    features['needs_refrigeration'] = 1 if re.search(r'\b(refrigerate|keep cold|chilled)\b', text_lower) else 0
    features['is_resealable'] = 1 if re.search(r'\b(resealable|reseal|zip)\b', text_lower) else 0
    features['single_serve'] = 1 if re.search(r'\b(single serve|individual|portion|on[- ]the[- ]go)\b', text_lower) else 0
    features['is_powder'] = 1 if re.search(r'\b(powder|powdered)\b', text_lower) else 0
    features['is_concentrate'] = 1 if re.search(r'\b(concentrate|concentrated)\b', text_lower) else 0
    features['microwaveable'] = 1 if re.search(r'\b(microwave|microwaveable)\b', text_lower) else 0
    features['oven_safe'] = 1 if re.search(r'\b(oven|bake)\b', text_lower) else 0

    # ============ NUTRITIONAL INFO ============
    combined_text = features['bullet_points_text'] + " " + features['product_description']

    protein_match = re.search(r'(\d+)\s*g.*protein', combined_text, re.IGNORECASE)
    features['protein_grams'] = int(protein_match.group(1)) if protein_match else 0

    calorie_match = re.search(r'(\d+)\s*calor', combined_text, re.IGNORECASE)
    features['calories'] = int(calorie_match.group(1)) if calorie_match else 0

    fiber_match = re.search(r'(\d+)\s*g.*fiber', combined_text, re.IGNORECASE)
    features['fiber_grams'] = int(fiber_match.group(1)) if fiber_match else 0

    sugar_match = re.search(r'(\d+)\s*g.*sugar', combined_text, re.IGNORECASE)
    features['sugar_grams'] = int(sugar_match.group(1)) if sugar_match else 0

    fat_match = re.search(r'(\d+)\s*g.*fat', combined_text, re.IGNORECASE)
    features['fat_grams'] = int(fat_match.group(1)) if fat_match else 0

    carb_match = re.search(r'(\d+)\s*g.*(carb|carbohydrate)', combined_text, re.IGNORECASE)
    features['carb_grams'] = int(carb_match.group(1)) if carb_match else 0

    sodium_match = re.search(r'(\d+)\s*mg.*sodium', combined_text, re.IGNORECASE)
    features['sodium_mg'] = int(sodium_match.group(1)) if sodium_match else 0

    servings_match = re.search(r'(\d+)\s*serving', combined_text, re.IGNORECASE)
    features['num_servings'] = int(servings_match.group(1)) if servings_match else 0

    features['protein_to_calorie_ratio'] = features['protein_grams'] / features['calories'] if features['calories'] > 0 else 0
    features['fiber_to_calorie_ratio'] = features['fiber_grams'] / features['calories'] if features['calories'] > 0 else 0
    features['has_nutrition_info'] = 1 if any([features['protein_grams'], features['calories'], features['fiber_grams']]) else 0
    features['nutrition_completeness'] = sum([
        1 if features['protein_grams'] > 0 else 0,
        1 if features['calories'] > 0 else 0,
        1 if features['fiber_grams'] > 0 else 0,
        1 if features['fat_grams'] > 0 else 0,
        1 if features['carb_grams'] > 0 else 0
    ])

    features['is_low_calorie_per_serving'] = 1 if 0 < features['calories'] < 100 else 0
    features['is_high_calorie_per_serving'] = 1 if features['calories'] > 300 else 0

    # ============ FLAVOR DESCRIPTORS ============
    features['has_flavor_descriptor'] = 1 if re.search(r'\b(flavor|flavored|taste|tasty)\b', text_lower) else 0
    features['is_spicy'] = 1 if re.search(r'\b(spicy|hot|jalapeno|chili|pepper|habanero|sriracha)\b', text_lower) else 0
    features['is_sweet'] = 1 if re.search(r'\b(sweet|sweetened|honey|sugar|syrup)\b', text_lower) else 0
    features['is_salty'] = 1 if re.search(r'\b(salty|salted|salt)\b', text_lower) else 0
    features['is_sour'] = 1 if re.search(r'\b(sour|tart|tangy)\b', text_lower) else 0
    features['is_savory'] = 1 if re.search(r'\b(savory|savoury|umami)\b', text_lower) else 0
    features['is_bitter'] = 1 if re.search(r'\b(bitter|dark)\b', text_lower) else 0
    features['is_mild'] = 1 if re.search(r'\b(mild|light)\b', text_lower) else 0
    features['is_bold'] = 1 if re.search(r'\b(bold|strong|intense)\b', text_lower) else 0
    features['is_smoky'] = 1 if re.search(r'\b(smok|bbq|barbecue)\b', text_lower) else 0

    flavor_count = sum([
        features['is_spicy'], features['is_sweet'], features['is_salty'],
        features['is_sour'], features['is_savory'], features['is_bitter']
    ])
    features['flavor_complexity'] = flavor_count

    # ============ INGREDIENT QUALITY ============
    features['real_ingredients'] = 1 if re.search(r'\b(real|100%|pure|authentic|genuine)\b', text_lower) else 0
    features['no_artificial'] = 1 if re.search(r'\b(no artificial|no preservatives|no additives|preservative free)\b', text_lower) else 0
    features['whole_grain'] = 1 if re.search(r'\b(whole grain|whole wheat|multigrain)\b', text_lower) else 0
    features['raw'] = 1 if re.search(r'\b(raw|unprocessed)\b', text_lower) else 0
    features['cold_pressed'] = 1 if re.search(r'\bcold[- ]pressed\b', text_lower) else 0
    features['first_press'] = 1 if re.search(r'\bfirst press\b', text_lower) else 0
    features['extra_virgin'] = 1 if re.search(r'\bextra virgin\b', text_lower) else 0
    features['unrefined'] = 1 if re.search(r'\bunrefined\b', text_lower) else 0
    features['grass_fed'] = 1 if re.search(r'\bgrass[- ]fed\b', text_lower) else 0
    features['free_range'] = 1 if re.search(r'\bfree[- ]range\b', text_lower) else 0

    ingredient_quality = sum([
        features['real_ingredients'], features['no_artificial'], features['whole_grain'],
        features['raw'], features['cold_pressed'], features['grass_fed']
    ])
    features['ingredient_quality_score'] = ingredient_quality

    # ============ TEXT ANALYSIS ============
    if features['item_name']:
        caps_count = sum(1 for c in features['item_name'] if c.isupper())
        features['name_caps_ratio'] = caps_count / len(features['item_name']) if len(features['item_name']) > 0 else 0
        features['name_has_all_caps_word'] = 1 if any(word.isupper() and len(word) > 1 for word in features['item_name'].split()) else 0
    else:
        features['name_caps_ratio'] = 0
        features['name_has_all_caps_word'] = 0

    number_matches = re.findall(r'\d+', features['item_name'])
    features['numbers_in_name'] = len(number_matches)
    features['has_year'] = 1 if re.search(r'\b(19|20)\d{2}\b', features['item_name']) else 0

    features['special_chars_in_name'] = sum(1 for c in features['item_name'] if c in ',-&()[]™®©')
    features['has_trademark'] = 1 if any(c in features['item_name'] for c in '™®') else 0
    features['has_parentheses'] = 1 if '(' in features['item_name'] else 0

    features['num_sentences'] = len(re.findall(r'[.!?]+', catalog_content))
    features['num_exclamations'] = catalog_content.count('!')
    features['num_questions'] = catalog_content.count('?')
    features['has_exclamation'] = 1 if features['num_exclamations'] > 0 else 0

    words = text_lower.split()
    if words:
        features['avg_word_length'] = np.mean([len(w) for w in words])
        features['max_word_length'] = max([len(w) for w in words])
        features['long_word_ratio'] = sum(1 for w in words if len(w) > 10) / len(words)
    else:
        features['avg_word_length'] = 0
        features['max_word_length'] = 0
        features['long_word_ratio'] = 0

    # ============ PREMIUM INDICATORS ============
    features['mentions_handpicked'] = 1 if re.search(r'\bhand[- ]picked\b', text_lower) else 0
    features['mentions_exclusive'] = 1 if re.search(r'\b(exclusive|limited edition|rare)\b', text_lower) else 0
    features['mentions_imported'] = 1 if re.search(r'\b(imported|international|exotic)\b', text_lower) else 0
    features['aged'] = 1 if re.search(r'\b(aged|matured)\b', text_lower) else 0

    premium_indicators = [
        features['has_premium_keywords'], features['is_organic'], features['quality_score'] > 0,
        features['mentions_exclusive'], features['is_handmade'], features['mentions_imported'],
        features['total_dietary_claims'] >= 2, features['ingredient_quality_score'] >= 2
    ]
    features['premium_score'] = sum(premium_indicators)
    features['is_luxury_product'] = 1 if features['premium_score'] >= 4 else 0

    return features

print("✓ Enhanced feature extraction function defined (150+ features)")


✓ Libraries imported successfully

✓ GPU DETECTED
✓ Enhanced feature extraction function defined (150+ features)


In [3]:

# ============================================================================
# STEP 4: Data Processing Function
# ============================================================================

def process_raw_data(df):
    """Process raw data and extract enhanced features"""
    print(f"Processing {len(df)} records...")

    extracted_features = []
    for idx, row in df.iterrows():
        if idx % 5000 == 0 and idx > 0:
            print(f"  Processed: {idx}/{len(df)} records ({idx/len(df)*100:.1f}%)")

        try:
            features = extract_enhanced_features(row['catalog_content'])
            features['sample_id'] = row['sample_id']
            # Add price for training data
            if 'price' in row:
                features['price'] = row['price']
            extracted_features.append(features)
        except Exception as e:
            print(f"  Warning: Error processing row {idx}: {str(e)}")
            features = {'sample_id': row['sample_id']}
            if 'price' in row:
                features['price'] = row['price']
            extracted_features.append(features)

    df_processed = pd.DataFrame(extracted_features)
    print(f"✓ Extracted {len(df_processed.columns)} features")

    return df_processed

print("✓ Data processing function defined")

# ============================================================================
# STEP 5: Feature Preparation with Leakage Detection
# ============================================================================

def prepare_features(df, label_encoders=None, scaler=None, feature_names=None, is_training=False):
    """
    Prepare features for modeling with leakage detection
    """
    # CRITICAL: Remove price-derived features to prevent leakage
    leakage_keywords = ['price_per', 'log_price', 'price_ratio',
                       'price_efficiency', 'price_normalized']

    # Remove non-feature columns
    non_feature_cols = [
        'sample_id', 'price', 'catalog_content', 'product_description',
        'bullet_points_text', 'item_name', 'brand', 'image_link'
    ]

    cols_to_drop = [c for c in non_feature_cols if c in df.columns]

    # Add leakage features to drop list
    for col in df.columns:
        if any(keyword in col.lower() for keyword in leakage_keywords):
            if col not in cols_to_drop:
                cols_to_drop.append(col)

    X = df.drop(columns=cols_to_drop, errors='ignore')

    # Handle missing values
    for col in X.columns:
        if X[col].isnull().any():
            if pd.api.types.is_numeric_dtype(X[col]):
                X[col].fillna(0, inplace=True)
            else:
                X[col].fillna('Unknown', inplace=True)

    # Encode categorical variables
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    if is_training:
        label_encoders = {}
        for col in cat_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le

        # Store feature names
        feature_names = X.columns.tolist()

        # Scale numerical features
        scaler = StandardScaler()
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

        return X, label_encoders, scaler, feature_names
    else:
        # For prediction: apply saved encoders and scaler
        for col in cat_cols:
            if col in label_encoders:
                le = label_encoders[col]
                X[col] = X[col].astype(str).apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
            else:
                X[col] = 0

        # Ensure all expected features are present
        for feature in feature_names:
            if feature not in X.columns:
                X[feature] = 0

        # Reorder columns to match training
        X = X[feature_names]

        # Scale using trained scaler
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X[numeric_cols] = scaler.transform(X[numeric_cols])

        return X

print("✓ Feature preparation function defined")

# ============================================================================
# STEP 6: SMAPE Metric and GPU-Optimized Training
# ============================================================================

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denominator = np.where(denominator == 0, 1e-10, denominator)
    return np.mean(np.abs(y_true - y_pred) / denominator) * 100

def train_ensemble_models(X_train, y_train, n_folds=5, use_gpu=True):
    """
    Train ensemble of XGBoost, LightGBM, and CatBoost with GPU support
    FIXED: CatBoost subsample issue resolved
    """
    print(f"\n{'='*70}")
    print("TRAINING ENSEMBLE MODELS WITH GPU ACCELERATION")
    print(f"{'='*70}")
    print(f"GPU Enabled: {use_gpu}")
    print(f"Training samples: {len(X_train)}")
    print(f"Features: {X_train.shape[1]}")

    y_log = np.log1p(y_train)
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    xgb_models, lgb_models, cb_models = [], [], []
    xgb_oof = np.zeros(len(y_train))
    lgb_oof = np.zeros(len(y_train))
    cb_oof = np.zeros(len(y_train))

    # GPU-Optimized Model Parameters
    xgb_params = {
        'objective': 'reg:squarederror',
        'learning_rate': 0.05,
        'max_depth': 8,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'seed': 42,
        'tree_method': 'gpu_hist' if use_gpu else 'hist',
        'predictor': 'gpu_predictor' if use_gpu else 'cpu_predictor',
        'verbosity': 0
    }

    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 40,
        'max_depth': 8,
        'min_child_samples': 20,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'device': 'gpu' if use_gpu else 'cpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'verbosity': -1,
        'n_jobs': -1
    }

    # FIXED: Added bootstrap_type for CatBoost subsample to work
    cb_params = {
        'loss_function': 'RMSE',
        'learning_rate': 0.05,
        'depth': 8,
        'l2_leaf_reg': 3,
        'bootstrap_type': 'Bernoulli',  # CRITICAL FIX: Required for subsample
        'subsample': 0.8,
        'random_seed': 42,
        'task_type': 'GPU' if use_gpu else 'CPU',
        'devices': '0' if use_gpu else None,
        'verbose': False,
        'thread_count': -1 if not use_gpu else None
    }

    print(f"\nRunning {n_folds}-Fold Cross-Validation...\n")

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train), 1):
        print(f"{'='*70}")
        print(f"Fold {fold}/{n_folds}:")
        print(f"{'='*70}")

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_log[train_idx], y_log[val_idx]
        y_val_actual = y_train[val_idx]

        # XGBoost with GPU
        print("  Training XGBoost...")
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=2000,
                              evals=[(dval, 'val')],
                              early_stopping_rounds=100,
                              verbose_eval=False)
        xgb_pred = np.expm1(xgb_model.predict(dval))
        xgb_oof[val_idx] = xgb_pred
        xgb_models.append(xgb_model)
        xgb_smape = smape(y_val_actual, xgb_pred)
        print(f"    ✓ XGBoost SMAPE: {xgb_smape:.4f}% | Best iteration: {xgb_model.best_iteration}")

        # LightGBM with GPU
        print("  Training LightGBM...")
        lgb_model = lgb.LGBMRegressor(**lgb_params, n_estimators=2000)
        lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_pred = np.expm1(lgb_model.predict(X_val))
        lgb_oof[val_idx] = lgb_pred
        lgb_models.append(lgb_model)
        lgb_smape = smape(y_val_actual, lgb_pred)
        print(f"    ✓ LightGBM SMAPE: {lgb_smape:.4f}% | Best iteration: {lgb_model.best_iteration_}")

        # CatBoost with GPU (FIXED)
        print("  Training CatBoost...")
        cb_model = cb.CatBoostRegressor(**cb_params, iterations=2000)
        cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                    early_stopping_rounds=100, verbose=False)
        cb_pred = np.expm1(cb_model.predict(X_val))
        cb_oof[val_idx] = cb_pred
        cb_models.append(cb_model)
        cb_smape = smape(y_val_actual, cb_pred)
        print(f"    ✓ CatBoost SMAPE: {cb_smape:.4f}% | Best iteration: {cb_model.best_iteration_}\n")

    # Overall OOF scores
    print(f"{'='*70}")
    print("OVERALL CROSS-VALIDATION SCORES")
    print(f"{'='*70}")
    xgb_oof_smape = smape(y_train, xgb_oof)
    lgb_oof_smape = smape(y_train, lgb_oof)
    cb_oof_smape = smape(y_train, cb_oof)
    print(f"XGBoost OOF SMAPE:  {xgb_oof_smape:.4f}%")
    print(f"LightGBM OOF SMAPE: {lgb_oof_smape:.4f}%")
    print(f"CatBoost OOF SMAPE: {cb_oof_smape:.4f}%")

    # Train meta-model (Ridge regression on OOF predictions)
    print(f"\n{'='*70}")
    print("TRAINING META-MODEL")
    print(f"{'='*70}")
    meta_train = np.column_stack([xgb_oof, lgb_oof, cb_oof])
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_train, y_log)
    meta_pred_log = meta_model.predict(meta_train)
    meta_pred = np.expm1(meta_pred_log)
    meta_smape = smape(y_train, meta_pred)
    print(f"✓ Meta-Model OOF SMAPE: {meta_smape:.4f}%")

    # Store models
    models_dict = {
        'xgb_models': xgb_models,
        'lgb_models': lgb_models,
        'cb_models': cb_models,
        'meta_model': meta_model
    }

    return models_dict

print("✓ Training function defined")

# ============================================================================
# STEP 7: Main Training Pipeline
# ============================================================================

print("\n" + "="*70)
print("STARTING TRAINING PIPELINE")
print("="*70)

# Load training data
print("\nLoading training data...")
train_file = 'train.csv'  # Change this to your training file path
df_train = pd.read_csv(train_file)
print(f"✓ Loaded {len(df_train)} training records")

# Process data
print(f"\n{'='*70}")
print("EXTRACTING FEATURES")
print(f"{'='*70}")
df_processed = process_raw_data(df_train)

# Prepare features
print(f"\n{'='*70}")
print("PREPARING FEATURES FOR TRAINING")
print(f"{'='*70}")

y_train = df_processed['price'].values
X_train, label_encoders, scaler, feature_names = prepare_features(
    df_processed, is_training=True
)

print(f"\n✓ Training data shape: {X_train.shape}")
print(f"✓ Number of features: {X_train.shape[1]}")
print(f"✓ Target variable: {len(y_train)} samples")

# Train models
models_dict = train_ensemble_models(X_train, y_train, n_folds=5, use_gpu=GPU_AVAILABLE)

# Save models
print(f"\n{'='*70}")
print("SAVING TRAINED MODELS")
print(f"{'='*70}")

joblib.dump(models_dict, 'trained_models.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_names, 'feature_names.pkl')

print("✓ Saved trained_models.pkl")
print("✓ Saved label_encoders.pkl")
print("✓ Saved scaler.pkl")
print("✓ Saved feature_names.pkl")

print(f"\n{'='*70}")
print("✅ TRAINING COMPLETE!")
print(f"{'='*70}")
print("\nYou can now use these files for prediction.")


✓ Data processing function defined
✓ Feature preparation function defined
✓ Training function defined

STARTING TRAINING PIPELINE

Loading training data...
✓ Loaded 75000 training records

EXTRACTING FEATURES
Processing 75000 records...
  Processed: 5000/75000 records (6.7%)
  Processed: 10000/75000 records (13.3%)
  Processed: 15000/75000 records (20.0%)
  Processed: 20000/75000 records (26.7%)
  Processed: 25000/75000 records (33.3%)
  Processed: 30000/75000 records (40.0%)
  Processed: 35000/75000 records (46.7%)
  Processed: 40000/75000 records (53.3%)
  Processed: 45000/75000 records (60.0%)
  Processed: 50000/75000 records (66.7%)
  Processed: 55000/75000 records (73.3%)
  Processed: 60000/75000 records (80.0%)
  Processed: 65000/75000 records (86.7%)
  Processed: 70000/75000 records (93.3%)
✓ Extracted 197 features

PREPARING FEATURES FOR TRAINING

✓ Training data shape: (75000, 191)
✓ Number of features: 191
✓ Target variable: 75000 samples

TRAINING ENSEMBLE MODELS WITH GPU AC

In [4]:
# ============================================================================
# OPTIMIZED PRICE PREDICTION - PREDICTION PIPELINE (FIXED VERSION)
# ============================================================================

# STEP 1: Install Required Libraries
!pip install pandas numpy scikit-learn joblib
!pip install xgboost lightgbm catboost

# ============================================================================
# STEP 2: Import Libraries
# ============================================================================

import pandas as pd
import numpy as np
import re
import joblib
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

# Check GPU
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], encoding='utf-8')
    print("\n✓ GPU DETECTED")
    GPU_AVAILABLE = True
except:
    print("\n⚠ No GPU detected. Using CPU mode.")
    GPU_AVAILABLE = False

# ============================================================================
# STEP 3: Load Trained Models
# ============================================================================

print("\n" + "="*70)
print("LOADING MODELS")
print("="*70)

models_dict = joblib.load('trained_models.pkl')
label_encoders = joblib.load('label_encoders.pkl')
scaler = joblib.load('scaler.pkl')
feature_names = joblib.load('feature_names.pkl')

print(f"✓ Loaded ensemble with {len(models_dict['xgb_models'])} fold models")
print(f"✓ Loaded {len(label_encoders)} label encoders")
print(f"✓ Loaded feature scaler")
print(f"✓ Expected {len(feature_names)} features")

# ============================================================================
# STEP 4: SAME Feature Extraction Function (MUST BE IDENTICAL TO TRAINING)
# ============================================================================

# Copy the ENTIRE extract_enhanced_features() function from Part 1
# This ensures identical feature extraction between training and prediction

# [INSERT COMPLETE extract_enhanced_features FUNCTION FROM PART 1 HERE]

# ============================================================================
# STEP 5: Processing Functions
# ============================================================================

def process_raw_data(df):
    """Process raw data and extract features"""
    print(f"Processing {len(df)} records...")

    extracted_features = []
    for idx, row in df.iterrows():
        if idx % 5000 == 0 and idx > 0:
            print(f"  Processed: {idx}/{len(df)} records ({idx/len(df)*100:.1f}%)")

        try:
            features = extract_enhanced_features(row['catalog_content'])
            features['sample_id'] = row['sample_id']
            extracted_features.append(features)
        except Exception as e:
            print(f"  Warning: Error processing row {idx}: {str(e)}")
            features = {'sample_id': row['sample_id']}
            extracted_features.append(features)

    df_processed = pd.DataFrame(extracted_features)
    print(f"✓ Extracted {len(df_processed.columns)} features")

    return df_processed

def prepare_features(df, label_encoders, scaler, feature_names):
    """Prepare features for prediction - FIXED VERSION"""
    # CRITICAL: Remove price-derived features (leakage prevention)
    leakage_keywords = ['price_per', 'log_price', 'price_ratio',
                       'price_efficiency', 'price_normalized']

    # Remove non-feature columns
    non_feature_cols = [
        'sample_id', 'catalog_content', 'product_description',
        'bullet_points_text', 'item_name', 'brand', 'image_link', 'price'
    ]

    cols_to_drop = [c for c in non_feature_cols if c in df.columns]

    # Add any leakage features found
    for col in df.columns:
        if any(keyword in col.lower() for keyword in leakage_keywords):
            if col not in cols_to_drop:
                cols_to_drop.append(col)

    X = df.drop(columns=cols_to_drop, errors='ignore')

    # Handle missing values
    for col in X.columns:
        if X[col].isnull().any():
            if pd.api.types.is_numeric_dtype(X[col]):
                X[col].fillna(0, inplace=True)
            else:
                X[col].fillna('Unknown', inplace=True)

    # Encode categorical variables
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    for col in cat_cols:
        if col in label_encoders:
            le = label_encoders[col]
            X[col] = X[col].astype(str).apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )
        else:
            X[col] = 0

    # Ensure all expected features are present
    for feature in feature_names:
        if feature not in X.columns:
            X[feature] = 0

    # Reorder columns to match training
    X = X[feature_names]

    # Scale using trained scaler
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    X[numeric_cols] = scaler.transform(X[numeric_cols])

    return X

def predict_with_ensemble(models_dict, X_test):
    """Generate predictions using ensemble"""
    print("\n" + "="*70)
    print("GENERATING PREDICTIONS")
    print("="*70)

    # Average predictions from all folds
    print("  Running XGBoost predictions...")
    xgb_preds = []
    for model in models_dict['xgb_models']:
        dtest = xgb.DMatrix(X_test)
        pred = np.expm1(model.predict(dtest))
        xgb_preds.append(pred)
    xgb_avg = np.mean(xgb_preds, axis=0)
    print(f"    ✓ XGBoost complete")

    print("  Running LightGBM predictions...")
    lgb_preds = []
    for model in models_dict['lgb_models']:
        pred = np.expm1(model.predict(X_test))
        lgb_preds.append(pred)
    lgb_avg = np.mean(lgb_preds, axis=0)
    print(f"    ✓ LightGBM complete")

    print("  Running CatBoost predictions...")
    cb_preds = []
    for model in models_dict['cb_models']:
        pred = np.expm1(model.predict(X_test))
        cb_preds.append(pred)
    cb_avg = np.mean(cb_preds, axis=0)
    print(f"    ✓ CatBoost complete")

    # Meta-model prediction
    print("  Running ensemble meta-model...")
    meta_test = np.column_stack([xgb_avg, lgb_avg, cb_avg])
    final_pred_log = models_dict['meta_model'].predict(meta_test)
    final_predictions = np.expm1(final_pred_log)

    # Ensure positive predictions
    final_predictions = np.maximum(final_predictions, 0.01)

    print(f"\n✓ Predictions complete!")
    print(f"  Range: ${final_predictions.min():.2f} - ${final_predictions.max():.2f}")
    print(f"  Mean:   ${final_predictions.mean():.2f}")
    print(f"  Median: ${np.median(final_predictions):.2f}")

    return final_predictions

print("✓ Processing and prediction functions defined")

# ============================================================================
# STEP 6: Load and Process Test Data
# ============================================================================

print("\n" + "="*70)
print("PROCESSING TEST DATA")
print("="*70)

# Load test data
test_filename = "test.csv"  # Change to your test file name
df_test = pd.read_csv(test_filename)
print(f"✓ Loaded {len(df_test)} test records")
print(f"✓ Columns: {list(df_test.columns)}")

# Extract features
print(f"\n{'='*70}")
print("EXTRACTING FEATURES")
print(f"{'='*70}")

df_processed = process_raw_data(df_test)

# Prepare features
print(f"\n{'='*70}")
print("PREPARING FEATURES FOR PREDICTION")
print(f"{'='*70}")

X_test = prepare_features(df_processed, label_encoders, scaler, feature_names)

print(f"\n✓ Test data shape: {X_test.shape}")
print(f"✓ Number of features: {X_test.shape[1]}")

# ============================================================================
# STEP 7: Generate Predictions
# ============================================================================

predictions = predict_with_ensemble(models_dict, X_test)

# ============================================================================
# STEP 8: Create Output File
# ============================================================================

print("\n" + "="*70)
print("CREATING OUTPUT FILE")
print("="*70)

output_df = pd.DataFrame({
    'sample_id': df_processed['sample_id'].values,
    'price': predictions
})

# Round to 2 decimal places
output_df['price'] = output_df['price']

output_df.to_csv('output.csv', index=False)

print(f"✓ Output saved to: output.csv")
print(f"✓ Total predictions: {len(output_df)}")

# Display sample predictions
print(f"\nFirst 20 predictions:")
print(output_df.head(20).to_string(index=False))

# Statistics
print(f"\n{'='*70}")
print("PREDICTION STATISTICS")
print(f"{'='*70}")
print(f"Count:  {len(predictions)}")
print(f"Min:    ${predictions.min():.2f}")
print(f"Max:    ${predictions.max():.2f}")
print(f"Mean:   ${predictions.mean():.2f}")
print(f"Median: ${np.median(predictions):.2f}")
print(f"Std:    ${predictions.std():.2f}")

# Price distribution
print(f"\nPrice Distribution:")
print(f"  $0-$10:      {sum(predictions < 10):,} ({sum(predictions < 10)/len(predictions)*100:.1f}%)")
print(f"  $10-$25:     {sum((predictions >= 10) & (predictions < 25)):,} ({sum((predictions >= 10) & (predictions < 25))/len(predictions)*100:.1f}%)")
print(f"  $25-$50:     {sum((predictions >= 25) & (predictions < 50)):,} ({sum((predictions >= 25) & (predictions < 50))/len(predictions)*100:.1f}%)")
print(f"  $50-$100:    {sum((predictions >= 50) & (predictions < 100)):,} ({sum((predictions >= 50) & (predictions < 100))/len(predictions)*100:.1f}%)")
print(f"  $100+:       {sum(predictions >= 100):,} ({sum(predictions >= 100)/len(predictions)*100:.1f}%)")

print("\n" + "="*70)
print("✅ PREDICTION COMPLETE!")
print("="*70)
print("\nYour predictions are ready in output.csv")


✓ Libraries imported successfully

✓ GPU DETECTED

LOADING MODELS
✓ Loaded ensemble with 5 fold models
✓ Loaded 2 label encoders
✓ Loaded feature scaler
✓ Expected 191 features
✓ Processing and prediction functions defined

PROCESSING TEST DATA
✓ Loaded 75000 test records
✓ Columns: ['sample_id', 'catalog_content', 'image_link']

EXTRACTING FEATURES
Processing 75000 records...
  Processed: 5000/75000 records (6.7%)
  Processed: 10000/75000 records (13.3%)
  Processed: 15000/75000 records (20.0%)
  Processed: 20000/75000 records (26.7%)
  Processed: 25000/75000 records (33.3%)
  Processed: 30000/75000 records (40.0%)
  Processed: 35000/75000 records (46.7%)
  Processed: 40000/75000 records (53.3%)
  Processed: 45000/75000 records (60.0%)
  Processed: 50000/75000 records (66.7%)
  Processed: 55000/75000 records (73.3%)
  Processed: 60000/75000 records (80.0%)
  Processed: 65000/75000 records (86.7%)
  Processed: 70000/75000 records (93.3%)
✓ Extracted 196 features

PREPARING FEATURES FOR